# RecoveryAI: research walkthrough

**Primary question.** To what extent can patient characteristics available near hospital admission predict length of
stay in real, de-identified hospital data (UCI Diabetes 130-US Hospitals), and how reliably can modern ML models quantify uncertainty in those predictions?

**Secondary question.** Do advanced nonlinear models provide statistically meaningful improvement over simple
statistical and naive baselines?

This notebook reads the artefacts written by `python -m training.run_pipeline --budget full`. It does not retrain
anything, and it never touches the test set again.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
R = json.loads((ROOT / "validation/results/validation_report.json").read_text())
pd.set_option("display.precision", 3)
print(R["research_questions"]["primary"])

To what extent can patient characteristics available near hospital admission predict length of stay in a real, de-identified hospital dataset, and how reliably can modern machine-learning models quantify uncertainty in those predictions?


## 1. Data: the LOS distribution
Length_of_Stay is `time_in_hospital` (1–14 days). One stay per patient (the first); deaths and hospice transfers removed.

In [2]:
d = R["data"]
print({k: d[k] for k in ["rows_raw", "repeat_encounters_removed", "death_or_hospice_removed", "n_dev", "n_test", "los_mean", "los_median", "los_sd", "los_skew"]})
pd.DataFrame({k: pd.Series(v) for k, v in d["group_means"].items()}).T

{'rows_raw': 101766, 'repeat_encounters_removed': 30248, 'death_or_hospice_removed': 1545, 'n_dev': 55976, 'n_test': 13994, 'los_mean': 4.273259968557953, 'los_median': 3.0, 'los_sd': 2.9339006467848843, 'los_skew': 1.1791553431559616}


,>200,>300,>7%,>8%,AfricanAmerican,Asian,Caucasian,Circulatory,Diabetes,Digestive,...,Normal,Not tested,Oral agents only,Other,Other/Unknown,Referral,Respiratory,Transfer,Unknown,Urgent
Gender,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Admission Type,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4.188,NaN,NaN,NaN,NaN,4.451
Admission Source,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,4.081,4.086,NaN,5.124,NaN,NaN
Medical Condition,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.171,4.205,4.276,...,NaN,NaN,NaN,4.612,NaN,NaN,4.041,NaN,NaN,NaN
Medication,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.993,NaN,NaN,NaN,NaN,NaN,NaN,NaN
A1C Result,NaN,NaN,4.787,4.638,NaN,NaN,NaN,NaN,NaN,NaN,...,4.845,4.170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Glucose Serum,4.666,5.274,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.971,4.265,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Race,NaN,NaN,NaN,NaN,4.336,3.684,4.277,NaN,NaN,NaN,...,NaN,NaN,NaN,4.161,NaN,NaN,NaN,NaN,4.239,NaN


![LOS](../validation/figures/los_distribution.png)

## 2. Nested cross-validation (headline generalisation estimate)

In [3]:
rows = []
for k, v in R["nested_cv"].items():
    s = v["summary"]
    rows.append({"model": k, "MAE mean": s["mae"]["mean"], "MAE sd": s["mae"]["sd"], "MAE median": s["mae"]["median"],
                 "IQR": s["mae"]["iqr"], "RMSE": s["rmse"]["mean"], "R2": s["r2"]["mean"],
                 "repeated-CV MAE": R["repeated_cv"][k]["summary"]["mae"]["mean"]})
pd.DataFrame(rows).set_index("model")

,MAE mean,MAE sd,MAE median,IQR,RMSE,R2,repeated-CV MAE
model,,,,,,,
median_baseline,2.221,0.019,2.223,0.034,3.200,-1.887e-01,2.224
condition_median,2.191,0.014,2.193,0.021,3.074,-9.674e-02,2.192
gamma_glm,2.133,0.010,2.134,0.014,2.912,1.544e-02,2.133
xgboost,2.103,0.013,2.105,0.011,2.889,3.118e-02,2.098
xgboost_aft,2.105,0.015,2.110,0.017,2.934,8.540e-04,2.103
catboost,2.104,0.016,2.103,0.013,2.921,9.607e-03,2.101
ft_transformer,2.104,0.010,2.109,0.014,2.916,1.294e-02,2.103


## 3. Model selection (CV only, locked before testing)

In [4]:
sel = R["selection"]; print(sel["reason"])
pd.DataFrame(sel["comparisons_vs_leader"]).T

2.103-day nested-CV MAE makes 'xgboost' the point-estimate leader, but 4 of 7 models are statistically indistinguishable from it (corrected resampled t-test, p > 0.05, or within 0.022 days = 1% of baseline MAE). By parsimony the simplest of these, 'xgboost', is selected.


,mean_diff,t,p_value,df,naive_p_value,equivalent_to_best
median_baseline,0.126,18.456,0.0,14,0.0,False
condition_median,0.094,24.659,0.0,14,0.0,False
gamma_glm,0.035,16.632,0.0,14,0.0,False
xgboost,0.0,0.0,1.0,14,NaN,True
xgboost_aft,0.005,2.379,0.032,14,0.0,True
catboost,0.003,1.604,0.131,14,0.004,True
ft_transformer,0.005,2.38,0.032,14,0.0,True


## 4. Final test set: bootstrap CIs and paired differences

In [5]:
T = R["test"]
pd.DataFrame({k: {"MAE": b["metrics"]["mae"]["estimate"], "BCa low": b["metrics"]["mae"]["ci_bca"][0], "BCa high": b["metrics"]["mae"]["ci_bca"][1],
                  "RMSE": b["metrics"]["rmse"]["estimate"], "R2": b["metrics"]["r2"]["estimate"], "bias": b["metrics"]["bias"]["estimate"]}
              for k, b in T["bootstrap"].items()}).T

,MAE,BCa low,BCa high,RMSE,R2,bias
median_baseline,2.216,2.180,2.254,3.191,-0.188,-1.268
condition_median,2.190,2.155,2.225,3.071,-0.100,-0.898
gamma_glm,2.137,2.105,2.170,2.913,0.010,-0.617
xgboost,2.096,2.064,2.130,2.890,0.026,-0.667
xgboost_aft,2.098,2.064,2.132,2.922,0.004,-0.802
catboost,2.094,2.060,2.127,2.902,0.017,-0.719
ft_transformer,2.100,2.068,2.134,2.883,0.030,-0.621


In [6]:
pd.DataFrame({k: {"MAE diff": v["mae"]["difference"], "CI low": v["mae"]["ci_percentile"][0], "CI high": v["mae"]["ci_percentile"][1],
                  "p": v["mae"]["p_value"]} for k, v in T["paired_differences"].items() if k.endswith("median_baseline")}).T

,MAE diff,CI low,CI high,p
condition_median__vs__median_baseline,-0.026,-0.036,-0.016,0.0
gamma_glm__vs__median_baseline,-0.079,-0.092,-0.065,0.0
xgboost__vs__median_baseline,-0.119,-0.135,-0.103,0.0
xgboost_aft__vs__median_baseline,-0.118,-0.131,-0.104,0.0
catboost__vs__median_baseline,-0.122,-0.136,-0.107,0.0
ft_transformer__vs__median_baseline,-0.115,-0.131,-0.099,0.0


![comparison](../validation/figures/model_comparison.png)

## 5. Shuffled-target negative control and permutation p-values

In [7]:
pd.DataFrame({k: {kk: p[kk] for kk in ["observed_mae", "null_mean", "null_sd", "improvement_over_shuffled", "p_value", "n_permutations"]}
              for k, p in R["permutation_tests"].items()}).T

,observed_mae,null_mean,null_sd,improvement_over_shuffled,p_value,n_permutations
condition_median,2.188,2.222,0.019,0.035,0.030,200.0
gamma_glm,2.127,2.223,0.018,0.096,0.005,200.0
xgboost,2.089,2.223,0.019,0.134,0.024,40.0
xgboost_aft,2.095,2.221,0.022,0.126,0.048,20.0
catboost,2.094,2.220,0.023,0.126,0.048,20.0
ft_transformer,2.092,2.223,0.013,0.131,0.250,3.0


![shuffled](../validation/figures/shuffled_target.png)

## 6. Statistical baseline: Gamma GLM (AIC / AICc / BIC are valid only here)

In [8]:
sm = R["statistical_model"]; print(sm["information_criteria"]); print(sm["null_comparison"])
pd.DataFrame(sm["coefficients"]).set_index("term")[["coef", "std_err", "ci_low", "ci_high", "exp_coef", "p_value"]]

{'log_likelihood': -107879.73378741232, 'k_parameters': 31, 'n': 47579, 'aic': 215821.46757482464, 'aicc': 215821.5093019578, 'bic': 216093.34212458308, 'deviance': 21207.261788556047, 'pearson_chi2': 21496.730113202168, 'dispersion': 0.45209636613182513}
{'lr_statistic': 3161.944235871284, 'df': 29, 'p_value': 0.0, 'null_log_likelihood': -109460.70590534796, 'full_log_likelihood_common_dispersion': -107879.73378741232, 'full_log_likelihood_reported': -107879.73378741232, 'aic_null': 218925.41181069592, 'aic_full': 215821.46757482464, 'bic_null': 218942.95210422872, 'bic_full': 216093.34212458308, 'interpretation': 'AIC/BIC lower for the null model means the added predictors do not improve the penalised likelihood enough to justify their parameters.'}


,coef,std_err,ci_low,ci_high,exp_coef,p_value
term,,,,,,
Intercept,1.696,3.579e-02,1.626,1.767,5.454,0.000e+00
Gender[Male],-0.023,6.241e-03,-0.035,-0.011,0.977,2.220e-04
Admission Type[Emergency],0.026,1.235e-02,0.001,0.050,1.026,3.824e-02
Admission Type[Other/Unknown],0.011,1.534e-02,-0.019,0.042,1.012,4.548e-01
Admission Type[Urgent],0.060,1.072e-02,0.039,0.081,1.062,2.529e-08
Admission Source[Other/Unknown],-0.069,1.730e-02,-0.103,-0.035,0.933,6.975e-05
Admission Source[Referral],-0.051,1.070e-02,-0.071,-0.030,0.951,2.327e-06
Admission Source[Transfer],0.158,1.309e-02,0.132,0.184,1.171,1.454e-33
Medical Condition[Diabetes],0.047,1.293e-02,0.022,0.073,1.049,2.420e-04


## 7. Calibration, learning curves and conformal coverage

![calib](../validation/figures/calibration_learning_coverage.png)

## 8. Positive control and quality gate

In [9]:
pc = ROOT / "validation/results/positive_control.json"
if pc.exists():
    P = json.loads(pc.read_text()); print(P["conclusion"]); display(pd.DataFrame({k: {"MAE": v["mae"]["mean"], "R2": v["r2"]["mean"]} for k, v in P["cv"].items()}).T)
g = R["quality_gate"]; print(g["headline"])
pd.DataFrame([{"check": c["label"], "failed": c["failed"]} for c in g["checks"]])

The framework recovers a known, modest admission-time effect: models clearly beat the median baseline and the shuffled-target control, so the pipeline is sensitive to signal of this size.


,MAE,R2
median_baseline,2.033,-0.027
gamma_glm,1.873,0.141
xgboost,1.861,0.157
catboost,1.852,0.163


All methodology checks passed.


,check,failed
0,Advanced models fail to meaningfully beat the ...,False
1,Bootstrap intervals show no meaningful differe...,False
2,Shuffled-target models perform similarly to re...,False
3,Prediction intervals are excessively wide,False
4,Calibration is poor or predictions are non-dis...,False


## Conclusion

Read the quality gate together with the shuffled-target control, the set A vs set B comparison and the positive control.
On real data, admission-time variables carry a real but modest signal: models beat the median baseline and the shuffled
target, but explain only a small share of the variance. Most of the apparent accuracy of richer feature sets comes from
information gathered during the stay, which would leak the answer at admission time. The earlier synthetic dataset, kept
in `validation/results/synthetic_reference/`, contained no signal at all.